# pandas-基本功能

## 教學目標

這份教學的目標是介紹基本的 pandas 功能，並學習快速操作大量的資料。

## 適用對象

適用於有程式基礎，且擁有 python 和 `numpy` 基礎的學生。

若沒有先學過 Python，建議先補充 Python 入門語法。

若沒有先學過 `numpy`，請參考 [numpy-基本功能](./numpy-基本功能.ipynb) 教學。

## 原教材的執行時間與測試環境

以下是原教材的測試記錄，實際執行時間會因內容與環境而不同。

本教學全部執行時間約為 1.3843801021575928 秒。

|測試環境|名稱|
|-|-|
|主機板|X570 AORUS ELITE|
|處理器|AMD Ryzen 7 3700X 8-Core Processor|
|記憶體|Kingston KHX3200C16D4/16GX|
|硬碟|Seagate ST1000DM003-1ER1|
|顯示卡|GeForce RTX 2080|
|作業系統|Ubuntu 18.04 LTS|

套件安裝與教材導覽請參考 [使用說明](./README.md)。

## 大綱

- 簡介與安裝
- 讀取與檢查資料
- 選取資料與處理缺失值
- 衍生欄位、摘要與準備 X／y
- 練習

## 簡介

根據 [pandas 官方網站](https://pandas.pydata.org/)（v1.0）：

> pandas is a fast, powerful, flexible and easy to use open source data analysis and manipulation tool,
built on top of the Python programming language.
>
> pandas 是基於 python 開發的一個快速、強大、具有彈性且容易使用的開源資料分析與操作工具。

`pandas` 提供**讀取**或**輸出**以下資料格式的功能：

- CSV
- JSON
- HTML
- Microsoft Excel
- OpenDocument Excel
- SQL databases
- pickle
- Google BigQuery

讀取資料後，`pandas` 能夠對資料進行以下的操作：

- **索引（Index**）與**選取（Select）** 資料
- **合併（Merge）**、**結合（Join）** 與**串接（Concatenate）** 資料
- **統計（Statistic）** 與**視覺化（Visualization）**

## 安裝

透過 `pip` 安裝 `pandas`：

```sh
pip install pandas
```

## 1. 讀取與檢查

DataFrame 是表格：一列（row）是一筆樣本，一欄（column）是一個欄位；單一欄是 Series。
使用隨附的 **12 筆模擬氣象資料**，不用下載外部檔案。
從本 notebook 所在的 `simplified` 目錄執行。

In [ ]:
import pandas as pd

df = pd.read_csv('./data/weather_sample.csv')
display(df.head())
df.info()
display(df.describe())

**檢查重點：**資料有幾筆？數字欄位是否讀成數字？有沒有缺失或不合理的範圍？
`describe()` 的 count 不含缺失值，不能直接當作總樣本數。

## 2. 選欄位與篩樣本

`df[欄位列表]` 選特徵；`df.loc[條件, 欄位]` 選符合條件的資料。
`iloc` 按位置取值，例如 `df.iloc[:3]` 是前三筆。

In [ ]:
features = df[['Temperature', 'Humidity']]
warm_days = df.loc[df['Temperature'] > 22, ['Day', 'Temperature']]
print("特徵表 shape：", features.shape)
display(warm_days)

## 3. 缺失值：先觀察，再決定策略

`isna().sum()` 檢查各欄缺失筆數；`fillna()` 可填補，`dropna()` 可刪除缺失樣本。
本例用中位數填補氣溫，只示範操作；填補或刪除都可能改變資料分布。
**實際 ML 要先切分資料，再由訓練集學習填補值**，驗證／測試集使用同一個值。

In [ ]:
print(df.isna().sum())
clean = df.copy()
clean['Temperature'] = clean['Temperature'].fillna(df['Temperature'].median())
print("填補後剩餘缺失數：", clean['Temperature'].isna().sum())

對應練習：[第 3 題](./exercises.ipynb#exercise-3)。


## 4. 衍生欄位與摘要

簡單公式可直接套用整欄；需要修改資料時，使用明確的欄位指派或 `.loc`。
不必為每筆資料另寫函式。摘要用來理解資料與檢查處理結果。

In [ ]:
clean['Temperature_F'] = clean['Temperature'] * 9 / 5 + 32
display(clean[['Day', 'Temperature', 'Temperature_F']].head())
display(clean.groupby('Weather')['Temperature'].agg(['count', 'mean']))

## 5. 準備 X 與 y

`X` 是用來預測的特徵，`y` 是答案。兩者必須對應同一批樣本、相同順序。
假設任務是用氣溫與濕度預測是否下雨，答案 `Rain` 不可同時放進 `X`。
類別欄位的編碼與訓練／驗證切分在 scikit-learn 篇處理。

In [ ]:
X = clean[['Temperature', 'Humidity']]
y = clean['Rain']
print("X:", X.shape, "y:", y.shape)
print("轉成 NumPy 後：", X.to_numpy().shape)

### 根據缺失值選取樣本

原教材介紹 `notna()` 找出有資料的位置，以及用條件篩選資料。
多個條件可用 `&` 組合；篩選後再從同一張表選出特徵與答案，便能保留對應的索引。
`index` 可用來檢查樣本順序；分組的 `count` 會統計該欄非缺失的筆數。

In [ ]:
has_temperature = df['Temperature'].notna()
has_humidity = df['Humidity'].notna()
example_rows = df.loc[has_temperature & has_humidity]
print('篩選後索引：', example_rows.index)
print(example_rows.groupby('Rain')['Temperature'].agg(['count']))

對應練習：[第 4 題](./exercises.ipynb#exercise-4)。


## 練習：找出高於平均氣溫的日期（程式填空）

使用原始 df 的有效氣溫計算平均，找出嚴格高於平均的日期與氣溫。保留原列順序，平均及氣溫輸出保留兩位小數。

請補上 `____`，保留固定資料與所有 print，不可直接寫死預期結果。NumPy 題請先匯入 `numpy as np`；pandas 題沿用教材讀入、尚未填補的原始 `df`。

**預期輸出（逐行比對）：**

```text
mean_temperature: 21.64
days: [4, 5, 6, 10, 11, 12]
temperatures: [23.0, 24.5, 22.0, 22.5, 25.0, 23.5]
count: 6
```


In [ ]:
mean_temp = df['Temperature'].____()
above_mean = df.loc[df['Temperature'] > ____, [____, ____]]
print('mean_temperature:', round(float(mean_temp), 2))
print('days:', above_mean['Day'].tolist())
print('temperatures:', above_mean['Temperature'].round(2).tolist())
print('count:', len(____))
